# SpaceX Falcon 9 Data Wrangling


This notebook analyzes SpaceX Falcon 9 launch data to explore factors affecting first-stage landing success and build predictive models.


## Objective
- Review missing values and data types in the launch dataset.
- Create a binary landing-success label for modeling.
- Save the cleaned output to `data/dataset_part_2.csv`.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / "data").exists() and (candidate / "notebooks").exists()
)
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOKS_DIR))

import pandas as pd

from notebook_utils import data_path, ensure_binary_file


## Data Loading


In [2]:
DATASET_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
)
dataset_path = data_path("dataset_part_1.csv")
if not dataset_path.exists():
    dataset_path = ensure_binary_file("dataset_part_1.csv", DATASET_URL)

df = pd.read_csv(dataset_path)
df.head()


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## Data Cleaning


In [3]:
missing_values = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
pd.DataFrame({"missing_pct": missing_values, "dtype": df.dtypes})


,missing_pct,dtype
Block,0.000000,float64
BoosterVersion,0.000000,object
Date,0.000000,object
FlightNumber,0.000000,int64
Flights,0.000000,int64
GridFins,0.000000,bool
LandingPad,28.888889,object
Latitude,0.000000,float64
LaunchSite,0.000000,object
Legs,0.000000,bool


## Exploratory Data Analysis


In [4]:
launch_site_counts = df["LaunchSite"].value_counts()
orbit_counts = df["Orbit"].value_counts()
launch_site_counts, orbit_counts


(LaunchSite
 CCAFS SLC 40    55
 KSC LC 39A      22
 VAFB SLC 4E     13
 Name: count, dtype: int64,
 Orbit
 GTO      27
 ISS      21
 VLEO     14
 PO        9
 LEO       7
 SSO       5
 MEO       3
 ES-L1     1
 HEO       1
 SO        1
 GEO       1
 Name: count, dtype: int64)

In [5]:
landing_outcomes = df["Outcome"].value_counts()
landing_outcomes


Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

In [6]:
bad_outcomes = set(landing_outcomes.index[[1, 3, 5, 6, 7]])
df["Class"] = df["Outcome"].apply(lambda outcome: 0 if outcome in bad_outcomes else 1)
df[["Outcome", "Class"]].head()


,Outcome,Class
0,None None,0
1,None None,0
2,None None,0
3,False Ocean,0
4,None None,0


## Feature Engineering


In [7]:
df["Class"].mean()


np.float64(0.6666666666666666)

In [8]:
output_path = data_path("dataset_part_2.csv")
df.to_csv(output_path, index=False)
output_path


PosixPath('/Users/riyana/Desktop/spacex/data/dataset_part_2.csv')

## Key Findings


In [9]:
df.groupby("LaunchSite")["Class"].mean().sort_values(ascending=False)


LaunchSite
KSC LC 39A      0.772727
VAFB SLC 4E     0.769231
CCAFS SLC 40    0.600000
Name: Class, dtype: float64